In [ ]:
import os
from dotenv import load_dotenv
from huggingface_hub import login
from datasets import load_dataset
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import numpy as np
import random
import time
from openai import OpenAI
from litellm import completion
from IPython.display import display
from data_prep.parser import scrub, parse
from data_prep.items import Item
from collections import Counter
from groq import Groq
import json
from data_prep.batch import Batch
from data_prep.items import Item

load_dotenv(override=True)

In [ ]:
groq_api_key = os.getenv("GROQ_API_KEY")
if groq_api_key:
    print("GROQ_API_KEY is set.")
else:
    print("GROQ_API_KEY is not set.")

openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
if openrouter_api_key:
    print("OPENROUTER_API_KEY is set.")
else:
    print("OPENROUTER_API_KEY is not set.")

hf_token = os.environ['HF_TOKEN']
if hf_token:
    print("HuggingFace token found.")
else:
    print("No HuggingFace token found.")

login(hf_token, add_to_git_credential=True)

#------------------------------

openrouter_url = "https://openrouter.ai/api/v1"


In [ ]:
username = "leearum95"
dataset = f"{username}/items_raw_full"

train, val, test = Item.from_hub(dataset)

items = train + val + test
items = items[20000:]

print(f"Loaded {len(items):,} items")
print(items[0])

In [ ]:
for index, item in enumerate(items):
    item.id = index

In [ ]:
# print(items[2633])

# 2145
# 2455
# 2633

In [ ]:
SYSTEM_PROMPT = """Create a concise description of a product. Respond only in this format. Do not include part numbers.
Summary: 1 sentence description of the product, including key features, purpose use cases. Be concise and informative.
"""

In [ ]:
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": items[0].full}]
response = completion(messages=messages, model="gpt-4.1-nano")

print(response.choices[0].message.content)
print()
print(items[0].full)
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cost: {response._hidden_params['response_cost']*100:.3f} cents")

# Process summary in batch mode on OpenAI

In [ ]:
Batch.create(items)



In [ ]:
Batch.run()

In [ ]:
Batch.fetch()

In [ ]:
for index, item in enumerate(items):
    if not item.summary:
        print(index)

In [ ]:
for item in items:
    item.full = None
    item.id = None

In [ ]:
username = "leearum95"
full = f"{username}/items_full_5a"
# full = f"{username}/items_full_multilang"



train = items[:1000]
val = items[1000:2000]
test = items[2000:]

# print(len(train), len(val), len(test))

Item.push_to_hub(full, train, val, test)

In [ ]:

# running_statuses = {"validating", "in_progress", "finalizing", "cancelling"}

client = OpenAI()

for file in client.files.list():
    client.files.delete(file.id)
    print("deleted", file.id)

# for batch in client.batches.list(limit=100):
#     if batch.status in running_statuses:
#         print(batch.id, batch.status, batch.endpoint)


# batch_id = "batch_69f24e260f24819086239fbb36d1196a"   # replace with your actual batch id

# batch = client.batches.cancel(batch_id)

# print(batch.id)
# print(batch.status)

In [ ]:
client = OpenAI()
running_statuses = {"validating", "in_progress", "finalizing", "cancelling"}
for batch in client.batches.list(limit=100):
    if batch.status in running_statuses:
        print(batch.id, batch.status, batch.endpoint)



In [ ]:
# from datetime import datetime
# from zoneinfo import ZoneInfo
# tz = ZoneInfo("America/New_York")

# start = datetime(2026, 4, 29, 15, 4, tzinfo=tz)  # 1:30 PM
# end   = datetime(2026, 4, 29, 17, 45, tzinfo=tz)  # 2:45 PM

# start_ts = int(start.timestamp())
# end_ts = int(end.timestamp())

# for batch in client.batches.list(limit=100):
#     if batch.status == "failed" and batch.failed_at:
#         if start_ts <= batch.failed_at <= end_ts:
#             failed_time = datetime.fromtimestamp(batch.failed_at, tz)
#             created_time = datetime.fromtimestamp(batch.created_at, tz)

#             print("batch id:", batch.id)
#             print("failed at:", failed_time.strftime("%Y-%m-%d %I:%M %p"))
#             print("created at:", created_time.strftime("%Y-%m-%d %I:%M %p"))
#             print("endpoint:", batch.endpoint)
#             print()